# Procesamiento de cercania a botaderos de basura

Este notebook convierte los valores con coma decimal, recalcula los porcentajes exactos y exporta el resultado a `data/processed`.

Formulas equivalentes de Excel:
- `Porcentaje.Si`: `=Total.Si/(Total.Si+Total.No)`
- `Porcentaje.No`: `=Total.No/(Total.Si+Total.No)`

In [ ]:
from pathlib import Path

import pandas as pd

# Permite ejecutar el notebook desde la carpeta del repositorio o desde notebooks.
repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

input_path = repo_root / "data" / "raw" / "osb_detsocia-botaderosbasura.csv"
output_dir = repo_root / "data" / "processed"
output_path = output_dir / "osb_detsocia-botaderosbasura-procesado.csv"

df = pd.read_csv(input_path, dtype="string")
df.head()

In [ ]:
# La fuente usa coma como separador decimal dentro de los campos numericos.
numeric_columns = ["Total.Si", "Total.No"]
for column in numeric_columns:
    df[column] = (
        df[column]
        .str.strip()
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

df[numeric_columns].dtypes

In [ ]:
# Equivalente a las formulas de Excel: =F2/(F2+G2) y =G2/(F2+G2).
total = df["Total.Si"] + df["Total.No"]
df["Porcentaje.Si"] = df["Total.Si"] / total
df["Porcentaje.No"] = df["Total.No"] / total

# Verificacion: cada fila debe sumar exactamente 1.
assert ((df["Porcentaje.Si"] + df["Porcentaje.No"]) - 1).abs().max() < 1e-12
df[["Loc", "Total.Si", "Total.No", "Porcentaje.Si", "Porcentaje.No"]]

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False, encoding="utf-8-sig", decimal=".")
print(f"Archivo exportado: {output_path}")
print(f"Filas exportadas: {len(df)}")